# 06 — Evaluación cuantitativa y cualitativa

> **Objetivo.** Medir el modelo fine-tuneado con tres métricas complementarias y revisar ejemplos cualitativos.

## Por qué tres métricas

| Métrica | Captura | Limitación |
|---|---|---|
| **SacreBLEU** | Calidad n-grama global | Penaliza correcciones cortas (típicas de GEC) |
| **GLEU** (Google-BLEU para GEC) | Premia ediciones mínimas correctas | Menos estandarizada en literatura general |
| **Exact-Match** | % oraciones idénticas a referencia | Dura, binaria — útil como suelo |

En producción, la métrica de oro es **ERRANT** (alinea ediciones token a token), pero requiere parser de inglés y queda fuera del scope PoC.

## ¿Por qué BLEU sub-evalúa GEC?

```
entrada:    "He go to store."
referencia: "He went to the store."
sistema A:  "He went to the store."        # perfecto
sistema B:  "He goes to the store."        # cambia tiempo, también válido
sistema C:  "He go to store."              # no hace nada
```

C tiene BLEU sólido por preservar la mayoría de n-gramas, aunque no corrige nada. **GLEU** penaliza esto porque también mira la *oración fuente*: premia n-gramas que están en la referencia **pero no** en la fuente.

In [ ]:
# === CONFIG ===
BATCH_SIZE      = 8
NUM_BEAMS       = 4
MAX_GEN_LEN     = 128
N_QUALITATIVE   = 20      # ejemplos a inspeccionar manualmente
# ==============

import os, json, torch
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_from_disk
from transformers import AutoTokenizer, T5ForConditionalGeneration
import evaluate

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

ROOT      = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = ROOT / "artifacts" / "t5-gec-mac" / "final"
SPLITS    = ROOT / "artifacts" / "coedit_gec_splits"

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Modelo:", MODEL_DIR)

## 1. Cargar modelo + datos eval

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = T5ForConditionalGeneration.from_pretrained(str(MODEL_DIR)).to(device)
model.eval()

splits = load_from_disk(str(SPLITS))
eval_set = splits["test"]
print("Eval ejemplos:", len(eval_set))

## 2. Generar predicciones sobre el eval set

Usamos beam search con `num_beams=4`. En MPS conviene mantener el batch_size moderado para evitar OOM.

In [ ]:
from tqdm.auto import tqdm

PREFIX = "grammar: "

def batch_generate(texts, batch_size=BATCH_SIZE):
    out = []
    for i in tqdm(range(0, len(texts), batch_size)):
        chunk = [PREFIX + t for t in texts[i:i+batch_size]]
        enc = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            ids = model.generate(
                **enc,
                max_new_tokens=MAX_GEN_LEN,
                num_beams=NUM_BEAMS,
                early_stopping=True,
            )
        out.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return out

srcs = list(eval_set["src"])
refs = list(eval_set["tgt"])
preds = batch_generate(srcs)
print("Generadas:", len(preds))

## 3. Métricas

### SacreBLEU

In [ ]:
bleu = evaluate.load("sacrebleu")
bleu_score = bleu.compute(predictions=preds, references=[[r] for r in refs])
print("SacreBLEU:", round(bleu_score["score"], 2))

### GLEU (Google-BLEU)

`evaluate.load("google_bleu")` carga la implementación de NLTK envuelta. GLEU es el promedio armónico de n-gram precision y recall, computado sobre rangos cortos (n=1..4).

No necesita `source` para esta variante simple — la versión específica de GEC sí, pero la community la usa con la fórmula clásica como proxy.

In [ ]:
try:
    gleu = evaluate.load("google_bleu")
    gleu_score = gleu.compute(predictions=preds, references=[[r] for r in refs])
    gleu_value = round(gleu_score["google_bleu"] * 100, 2)
except Exception as e:
    print("google_bleu no disponible, fallback a NLTK directo:", e)
    from nltk.translate.gleu_score import corpus_gleu
    pred_tok = [p.split() for p in preds]
    ref_tok  = [[r.split()] for r in refs]
    gleu_value = round(corpus_gleu(ref_tok, pred_tok) * 100, 2)

print("GLEU:", gleu_value)

### Exact-Match

In [ ]:
def normalize(s: str) -> str:
    return " ".join(s.strip().lower().split())

em_count = sum(1 for p, r in zip(preds, refs) if normalize(p) == normalize(r))
em_score = em_count / len(preds) * 100
print(f"Exact-Match: {em_score:.2f}%   ({em_count}/{len(preds)})")

### Tasa de "no edita"

¿Con qué frecuencia el modelo deja `output == input`? Útil para entender si está siendo demasiado conservador.

In [ ]:
noop_count = sum(1 for s, p in zip(srcs, preds) if normalize(s) == normalize(p))
noop_rate  = noop_count / len(preds) * 100

# Fracción de eval donde la referencia tampoco edita (oración ya correcta)
ref_noop  = sum(1 for s, r in zip(srcs, refs) if normalize(s) == normalize(r))
ref_noop_rate = ref_noop / len(refs) * 100

print(f"Modelo deja sin editar    : {noop_rate:.2f}%")
print(f"Referencia deja sin editar: {ref_noop_rate:.2f}%")
print("(idealmente ambos cercanos)")

## 4. Evaluación cualitativa

Las métricas numéricas no cuentan toda la historia. Inspeccionamos ejemplos donde el modelo:

- **Acertó** (pred == ref).
- **Falló razonablemente** (pred != ref pero corrección plausible).
- **Falló mal** (pred no corrige, o introduce error).

In [ ]:
df_eval = pd.DataFrame({"src": srcs, "ref": refs, "pred": preds})
df_eval["src_norm"]  = df_eval["src"].map(normalize)
df_eval["ref_norm"]  = df_eval["ref"].map(normalize)
df_eval["pred_norm"] = df_eval["pred"].map(normalize)

df_eval["correct"]      = df_eval["pred_norm"] == df_eval["ref_norm"]
df_eval["src_is_noop"]   = df_eval["src_norm"] == df_eval["ref_norm"]
df_eval["pred_is_noop"]  = df_eval["src_norm"] == df_eval["pred_norm"]

print("Aciertos:", df_eval["correct"].sum(), "/", len(df_eval))

In [ ]:
print("=== ACIERTOS (muestra) ===\n")
for _, r in df_eval[df_eval["correct"]].head(N_QUALITATIVE//2).iterrows():
    print("SRC :", r["src"])
    print("PRED:", r["pred"])
    print()

In [ ]:
print("=== FALLOS (muestra) ===\n")
for _, r in df_eval[~df_eval["correct"]].head(N_QUALITATIVE//2).iterrows():
    print("SRC :", r["src"])
    print("REF :", r["ref"])
    print("PRED:", r["pred"])
    print()

## 5. Persistir métricas

In [ ]:
metrics = {
    "sacrebleu": round(bleu_score["score"], 4),
    "gleu": gleu_value,
    "exact_match_pct": round(em_score, 4),
    "model_noop_rate_pct": round(noop_rate, 4),
    "ref_noop_rate_pct": round(ref_noop_rate, 4),
    "n_eval": len(preds),
    "num_beams": NUM_BEAMS,
}

OUT = ROOT / "artifacts" / "eval_metrics.json"
OUT.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print("\nGuardado en:", OUT)

# También un CSV con todas las predicciones para análisis offline
df_eval[["src","ref","pred","correct"]].to_csv(ROOT / "artifacts" / "eval_predictions.csv", index=False)

## Resumen

Métricas reportadas en `artifacts/eval_metrics.json`:

- **SacreBLEU** — calidad global.
- **GLEU** — calidad orientada a GEC.
- **Exact-Match** — % de coincidencia exacta.
- **noop_rate** — diagnóstico de conservadurismo.

Ejemplos cualitativos en `artifacts/eval_predictions.csv`.

**Siguiente → `07_inferencia_confianza.ipynb`** — score de confianza vía logits.
